# Assignment 08: Neural Networks
### CS201L: Artificial Intelligence Laboratory
### Indian Institute of Technology, Dharwad

---
**Dataset:** Human Activity Recognition (HAR) using Smartphones  
**Task:** Multiclass classification of 6 human activities


## Imports and Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# so that results are somewhat reproducible
torch.manual_seed(42)
np.random.seed(42)

print("All imports done!")
print(f"PyTorch version: {torch.__version__}")

## Helper: Label Encoder (fit on training labels)

We need a common label encoder across all parts so class indices stay consistent.

In [ ]:
# We will fit the label encoder once on the original training data
# and reuse it everywhere
le = LabelEncoder()

# Fit it now using the original train file
_tmp = pd.read_csv('data/Activity Train.csv')
le.fit(_tmp['Activity'])
print("Classes:", list(le.classes_))

## Helper Functions (reused across all parts)

Writing everything as functions so we don't repeat code for each part.

In [ ]:
def load_data(train_path, val_path, test_path, target_col='Activity'):
    """
    Load csv files and convert to PyTorch tensors.
    Returns: (X_train, y_train, X_val, y_val, X_test, y_test)
    """
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)

    X_train = torch.tensor(train_df.drop(columns=[target_col]).values, dtype=torch.float32)
    X_val   = torch.tensor(val_df.drop(columns=[target_col]).values,   dtype=torch.float32)
    X_test  = torch.tensor(test_df.drop(columns=[target_col]).values,  dtype=torch.float32)

    y_train = torch.tensor(le.transform(train_df[target_col]), dtype=torch.long)
    y_val   = torch.tensor(le.transform(val_df[target_col]),   dtype=torch.long)
    y_test  = torch.tensor(le.transform(test_df[target_col]),  dtype=torch.long)

    print(f"Train size: {X_train.shape}, Val size: {X_val.shape}, Test size: {X_test.shape}")
    print(f"Input features: {X_train.shape[1]}")
    return X_train, y_train, X_val, y_val, X_test, y_test


def train_model(model, X_train, y_train, lr=0.01, max_epochs=1000,
                patience=10, threshold=1e-3):
    """
    Train a model using SGD + CrossEntropy.
    Returns list of training losses per epoch.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    loss_history = []
    prev_loss = float('inf')
    counter   = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss    = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        train_loss = loss.item()
        loss_history.append(train_loss)

        # convergence check
        if abs(train_loss - prev_loss) < threshold:
            counter += 1
        else:
            counter = 0
        prev_loss = train_loss

        if counter >= patience:
            print(f"  Converged at epoch {epoch} | Final loss: {train_loss:.4f}")
            break

        if epoch % 100 == 0:
            print(f"  Epoch {epoch}/{max_epochs} | Loss: {train_loss:.4f}")

    return loss_history


def evaluate_model(model, X, y):
    """
    Evaluate model, return accuracy and predictions.
    """
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        preds   = torch.argmax(outputs, dim=1)
    acc = accuracy_score(y.numpy(), preds.numpy())
    return acc, preds.numpy()


def print_metrics(y_true, y_pred, dataset_name="Test"):
    """
    Print confusion matrix + accuracy + micro/macro precision, recall, F1.
    """
    print(f"\n{'='*55}")
    print(f"  Metrics on {dataset_name} Data")
    print(f"{'='*55}")

    acc = accuracy_score(y_true, y_pred)
    print(f"  Accuracy: {acc*100:.2f}%")

    print("\n  Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)

    # plot confusion matrix nicely
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix - {dataset_name}')
    plt.tight_layout()
    plt.show()

    print("\n  Micro averages:")
    print(f"    Precision : {precision_score(y_true, y_pred, average='micro'):.4f}")
    print(f"    Recall    : {recall_score(y_true, y_pred, average='micro'):.4f}")
    print(f"    F1-score  : {f1_score(y_true, y_pred, average='micro'):.4f}")

    print("\n  Macro averages:")
    print(f"    Precision : {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"    Recall    : {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"    F1-score  : {f1_score(y_true, y_pred, average='macro'):.4f}")

    print("\n  Full Classification Report:")
    print(classification_report(y_true, y_pred, target_names=le.classes_))


def plot_loss_curves(loss_dict, title="Epoch vs Loss"):
    """
    Plot training loss curves for multiple architectures.
    loss_dict: {arch_label: [list of losses]}
    """
    plt.figure(figsize=(10, 5))
    for label, losses in loss_dict.items():
        plt.plot(losses, label=label)
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def save_models(models_dict, folder, prefix):
    """
    Save all models in a dictionary to a folder.
    models_dict: {label: model}
    """
    os.makedirs(folder, exist_ok=True)
    for label, model in models_dict.items():
        # make filename filesystem-safe
        safe_label = str(label).replace(' ', '_').replace('(', '').replace(')', '').replace(',', '_')
        path = os.path.join(folder, f"{prefix}_{safe_label}.pth")
        torch.save(model.state_dict(), path)
    print(f"  Saved {len(models_dict)} model(s) to '{folder}/'")


print("Helper functions defined!")

## Neural Network Classes

Separate classes for single hidden layer and two hidden layer networks as asked.

In [ ]:
class SingleHiddenLayerNN(nn.Module):
    """
    Neural network with ONE hidden layer.
    Architecture: Input -> Hidden (tanh) -> Output (softmax via CrossEntropyLoss)
    """
    def __init__(self, input_size, hidden_size, output_size=6):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),  # input layer (linear neurons)
            nn.Tanh(),                           # hidden layer activation
            nn.Linear(hidden_size, output_size)  # output layer (softmax applied by loss fn)
        )

    def forward(self, x):
        return self.layers(x)


class TwoHiddenLayerNN(nn.Module):
    """
    Neural network with TWO hidden layers.
    Architecture: Input -> Hidden1 (tanh) -> Hidden2 (tanh) -> Output
    """
    def __init__(self, input_size, h1, h2, output_size=6):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, h1),  # input to hidden layer 1
            nn.Tanh(),                  # activation for hidden layer 1
            nn.Linear(h1, h2),          # hidden layer 1 to hidden layer 2
            nn.Tanh(),                  # activation for hidden layer 2
            nn.Linear(h2, output_size)  # hidden layer 2 to output
        )

    def forward(self, x):
        return self.layers(x)


print("Model classes defined!")

---
# Part A: Original Dataset

Using the raw/original dataset without any preprocessing.

### A.1 Load Data

In [ ]:
print("Loading original dataset...")
X_train_A, y_train_A, X_val_A, y_val_A, X_test_A, y_test_A = load_data(
    'data/Activity Train.csv',
    'data/Activity Validation.csv',
    'data/Activity Test.csv'
)

INPUT_SIZE_A = X_train_A.shape[1]  # 561
OUTPUT_SIZE  = 6
print(f"Input size: {INPUT_SIZE_A}, Output size: {OUTPUT_SIZE}")

### A.2 Single Hidden Layer Architectures

In [ ]:
# Architecture configurations for single hidden layer
# (hidden_size,)
single_archs_A = {
    'Arch1 (561)' : 561,
    'Arch2 (1024)': 1024,
    'Arch3 (128)' : 128,
}

single_models_A    = {}   # store trained models
single_losses_A    = {}   # store loss curves
single_val_accs_A  = {}   # store validation accuracies

for arch_name, hidden_size in single_archs_A.items():
    print(f"\nTraining Single Hidden Layer - {arch_name}")
    model = SingleHiddenLayerNN(INPUT_SIZE_A, hidden_size, OUTPUT_SIZE)
    losses = train_model(model, X_train_A, y_train_A,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_A, y_val_A)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    single_models_A[arch_name]   = model
    single_losses_A[arch_name]   = losses
    single_val_accs_A[arch_name] = val_acc

# save models
save_models(single_models_A, 'saved_models/partA', 'single')

### A.3 Two Hidden Layer Architectures

In [ ]:
# Architecture configurations for two hidden layers
# (h1, h2)
two_archs_A = {
    'Arch1 (561,561)' : (561,  561),
    'Arch2 (1024,128)': (1024, 128),
    'Arch3 (1024,64)' : (1024,  64),
    'Arch4 (256,64)'  : (256,   64),
}

two_models_A    = {}
two_losses_A    = {}
two_val_accs_A  = {}

for arch_name, (h1, h2) in two_archs_A.items():
    print(f"\nTraining Two Hidden Layer - {arch_name}")
    model = TwoHiddenLayerNN(INPUT_SIZE_A, h1, h2, OUTPUT_SIZE)
    losses = train_model(model, X_train_A, y_train_A,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_A, y_val_A)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    two_models_A[arch_name]   = model
    two_losses_A[arch_name]   = losses
    two_val_accs_A[arch_name] = val_acc

# save models
save_models(two_models_A, 'saved_models/partA', 'two')

### A.4 Plot Loss Curves

In [ ]:
plot_loss_curves(single_losses_A, title="Part A - Single Hidden Layer: Epoch vs Loss (Original Data)")
plot_loss_curves(two_losses_A,    title="Part A - Two Hidden Layers: Epoch vs Loss (Original Data)")

### A.5 Validation Accuracy Summary and Best Model Selection

In [ ]:
print("--- Single Hidden Layer Validation Accuracies (Part A) ---")
for name, acc in single_val_accs_A.items():
    print(f"  {name}: {acc*100:.2f}%")

best_single_A = max(single_val_accs_A, key=single_val_accs_A.get)
print(f"\n  Best single hidden layer: {best_single_A} with val acc {single_val_accs_A[best_single_A]*100:.2f}%")

print("\n--- Two Hidden Layer Validation Accuracies (Part A) ---")
for name, acc in two_val_accs_A.items():
    print(f"  {name}: {acc*100:.2f}%")

best_two_A = max(two_val_accs_A, key=two_val_accs_A.get)
print(f"\n  Best two hidden layer: {best_two_A} with val acc {two_val_accs_A[best_two_A]*100:.2f}%")

### A.6 Test Set Evaluation (Best Models)

In [ ]:
print(f"Evaluating best single hidden layer model: {best_single_A}")
_, preds_single_A = evaluate_model(single_models_A[best_single_A], X_test_A, y_test_A)
print_metrics(y_test_A.numpy(), preds_single_A, dataset_name=f"Part A - {best_single_A} (Single Layer)")

print(f"\nEvaluating best two hidden layer model: {best_two_A}")
_, preds_two_A = evaluate_model(two_models_A[best_two_A], X_test_A, y_test_A)
print_metrics(y_test_A.numpy(), preds_two_A, dataset_name=f"Part A - {best_two_A} (Two Layers)")

---
# Part B: Standardized Dataset

### B.1 Load Data

In [ ]:
print("Loading standardized dataset...")
X_train_B, y_train_B, X_val_B, y_val_B, X_test_B, y_test_B = load_data(
    'data/Activity Scaled Train.csv',
    'data/Activity Scaled Validation.csv',
    'data/Activity Scaled Test.csv'
)

INPUT_SIZE_B = X_train_B.shape[1]  # 561
print(f"Input size: {INPUT_SIZE_B}")

### B.2 Single Hidden Layer Architectures

In [ ]:
single_archs_B = {
    'Arch1 (10)' : 10,
    'Arch2 (100)': 100,
    'Arch3 (64)' : 64,
    'Arch4 (512)': 512,
}

single_models_B   = {}
single_losses_B   = {}
single_val_accs_B = {}

for arch_name, hidden_size in single_archs_B.items():
    print(f"\nTraining Single Hidden Layer - {arch_name}")
    model = SingleHiddenLayerNN(INPUT_SIZE_B, hidden_size, OUTPUT_SIZE)
    losses = train_model(model, X_train_B, y_train_B,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_B, y_val_B)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    single_models_B[arch_name]   = model
    single_losses_B[arch_name]   = losses
    single_val_accs_B[arch_name] = val_acc

save_models(single_models_B, 'saved_models/partB', 'single')

### B.3 Two Hidden Layer Architectures

In [ ]:
two_archs_B = {
    'Arch1 (64,16)'  : (64,  16),
    'Arch2 (128,32)' : (128, 32),
    'Arch3 (256,64)' : (256, 64),
    'Arch4 (512,128)': (512, 128),
}

two_models_B   = {}
two_losses_B   = {}
two_val_accs_B = {}

for arch_name, (h1, h2) in two_archs_B.items():
    print(f"\nTraining Two Hidden Layer - {arch_name}")
    model = TwoHiddenLayerNN(INPUT_SIZE_B, h1, h2, OUTPUT_SIZE)
    losses = train_model(model, X_train_B, y_train_B,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_B, y_val_B)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    two_models_B[arch_name]   = model
    two_losses_B[arch_name]   = losses
    two_val_accs_B[arch_name] = val_acc

save_models(two_models_B, 'saved_models/partB', 'two')

### B.4 Plot Loss Curves

In [ ]:
plot_loss_curves(single_losses_B, title="Part B - Single Hidden Layer: Epoch vs Loss (Scaled Data)")
plot_loss_curves(two_losses_B,    title="Part B - Two Hidden Layers: Epoch vs Loss (Scaled Data)")

### B.5 Best Model Selection and Test Evaluation

In [ ]:
print("--- Single Hidden Layer Validation Accuracies (Part B) ---")
for name, acc in single_val_accs_B.items():
    print(f"  {name}: {acc*100:.2f}%")
best_single_B = max(single_val_accs_B, key=single_val_accs_B.get)
print(f"\n  Best: {best_single_B} -> {single_val_accs_B[best_single_B]*100:.2f}%")

print("\n--- Two Hidden Layer Validation Accuracies (Part B) ---")
for name, acc in two_val_accs_B.items():
    print(f"  {name}: {acc*100:.2f}%")
best_two_B = max(two_val_accs_B, key=two_val_accs_B.get)
print(f"\n  Best: {best_two_B} -> {two_val_accs_B[best_two_B]*100:.2f}%")

In [ ]:
_, preds_single_B = evaluate_model(single_models_B[best_single_B], X_test_B, y_test_B)
print_metrics(y_test_B.numpy(), preds_single_B, dataset_name=f"Part B - {best_single_B} (Single Layer)")

_, preds_two_B = evaluate_model(two_models_B[best_two_B], X_test_B, y_test_B)
print_metrics(y_test_B.numpy(), preds_two_B, dataset_name=f"Part B - {best_two_B} (Two Layers)")

---
# Part C: PCA Transformed Dataset (All Components)

### C.1 Load Data

In [ ]:
print("Loading PCA-All dataset...")
X_train_C, y_train_C, X_val_C, y_val_C, X_test_C, y_test_C = load_data(
    'data/Activity PCAAll Train.csv',
    'data/Activity PCA-All Validation.csv',
    'data/Activity PCAAll Test.csv'
)

INPUT_SIZE_C = X_train_C.shape[1]  # 561 (all PCA components)
print(f"Input size: {INPUT_SIZE_C}")

### C.2 Single Hidden Layer Architectures

In [ ]:
single_archs_C = {
    'Arch1 (10)' : 10,
    'Arch2 (100)': 100,
    'Arch3 (64)' : 64,
    'Arch4 (512)': 512,
}

single_models_C   = {}
single_losses_C   = {}
single_val_accs_C = {}

for arch_name, hidden_size in single_archs_C.items():
    print(f"\nTraining Single Hidden Layer - {arch_name}")
    model = SingleHiddenLayerNN(INPUT_SIZE_C, hidden_size, OUTPUT_SIZE)
    losses = train_model(model, X_train_C, y_train_C,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_C, y_val_C)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    single_models_C[arch_name]   = model
    single_losses_C[arch_name]   = losses
    single_val_accs_C[arch_name] = val_acc

save_models(single_models_C, 'saved_models/partC', 'single')

### C.3 Two Hidden Layer Architectures

In [ ]:
two_archs_C = {
    'Arch1 (64,16)'  : (64,  16),
    'Arch2 (128,32)' : (128, 32),
    'Arch3 (256,64)' : (256, 64),
    'Arch4 (512,128)': (512, 128),
}

two_models_C   = {}
two_losses_C   = {}
two_val_accs_C = {}

for arch_name, (h1, h2) in two_archs_C.items():
    print(f"\nTraining Two Hidden Layer - {arch_name}")
    model = TwoHiddenLayerNN(INPUT_SIZE_C, h1, h2, OUTPUT_SIZE)
    losses = train_model(model, X_train_C, y_train_C,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_C, y_val_C)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    two_models_C[arch_name]   = model
    two_losses_C[arch_name]   = losses
    two_val_accs_C[arch_name] = val_acc

save_models(two_models_C, 'saved_models/partC', 'two')

### C.4 Plot Loss Curves

In [ ]:
plot_loss_curves(single_losses_C, title="Part C - Single Hidden Layer: Epoch vs Loss (PCA All)")
plot_loss_curves(two_losses_C,    title="Part C - Two Hidden Layers: Epoch vs Loss (PCA All)")

### C.5 Best Model Selection and Test Evaluation

In [ ]:
print("--- Single Hidden Layer Validation Accuracies (Part C) ---")
for name, acc in single_val_accs_C.items():
    print(f"  {name}: {acc*100:.2f}%")
best_single_C = max(single_val_accs_C, key=single_val_accs_C.get)
print(f"\n  Best: {best_single_C} -> {single_val_accs_C[best_single_C]*100:.2f}%")

print("\n--- Two Hidden Layer Validation Accuracies (Part C) ---")
for name, acc in two_val_accs_C.items():
    print(f"  {name}: {acc*100:.2f}%")
best_two_C = max(two_val_accs_C, key=two_val_accs_C.get)
print(f"\n  Best: {best_two_C} -> {two_val_accs_C[best_two_C]*100:.2f}%")

In [ ]:
_, preds_single_C = evaluate_model(single_models_C[best_single_C], X_test_C, y_test_C)
print_metrics(y_test_C.numpy(), preds_single_C, dataset_name=f"Part C - {best_single_C} (Single Layer)")

_, preds_two_C = evaluate_model(two_models_C[best_two_C], X_test_C, y_test_C)
print_metrics(y_test_C.numpy(), preds_two_C, dataset_name=f"Part C - {best_two_C} (Two Layers)")

---
# Part D: PCA Transformed Dataset (99% Variance)

Note: Input size here is 156 (not 561) since only components explaining 99% variance are kept.

### D.1 Load Data

In [ ]:
print("Loading PCA-99 dataset...")
X_train_D, y_train_D, X_val_D, y_val_D, X_test_D, y_test_D = load_data(
    'data/Activity PCA99 Train.csv',
    'data/Activity PCA99 Validation.csv',
    'data/Activity PCA99 Test.csv'
)

INPUT_SIZE_D = X_train_D.shape[1]  # should be 156
print(f"Input size: {INPUT_SIZE_D}")

### D.2 Single Hidden Layer Architectures

In [ ]:
single_archs_D = {
    'Arch1 (156)': 156,
    'Arch2 (512)': 512,
    'Arch3 (64)' : 64,
}

single_models_D   = {}
single_losses_D   = {}
single_val_accs_D = {}

for arch_name, hidden_size in single_archs_D.items():
    print(f"\nTraining Single Hidden Layer - {arch_name}")
    model = SingleHiddenLayerNN(INPUT_SIZE_D, hidden_size, OUTPUT_SIZE)
    losses = train_model(model, X_train_D, y_train_D,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_D, y_val_D)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    single_models_D[arch_name]   = model
    single_losses_D[arch_name]   = losses
    single_val_accs_D[arch_name] = val_acc

save_models(single_models_D, 'saved_models/partD', 'single')

### D.3 Two Hidden Layer Architectures

In [ ]:
two_archs_D = {
    'Arch1 (156,156)': (156, 156),
    'Arch2 (512,64)' : (512,  64),
    'Arch3 (256,64)' : (256,  64),
    'Arch4 (64,32)'  : (64,   32),
}

two_models_D   = {}
two_losses_D   = {}
two_val_accs_D = {}

for arch_name, (h1, h2) in two_archs_D.items():
    print(f"\nTraining Two Hidden Layer - {arch_name}")
    model = TwoHiddenLayerNN(INPUT_SIZE_D, h1, h2, OUTPUT_SIZE)
    losses = train_model(model, X_train_D, y_train_D,
                         lr=0.01, max_epochs=1000, patience=10, threshold=1e-3)
    val_acc, _ = evaluate_model(model, X_val_D, y_val_D)
    print(f"  Validation Accuracy: {val_acc*100:.2f}%")

    two_models_D[arch_name]   = model
    two_losses_D[arch_name]   = losses
    two_val_accs_D[arch_name] = val_acc

save_models(two_models_D, 'saved_models/partD', 'two')

### D.4 Plot Loss Curves

In [ ]:
plot_loss_curves(single_losses_D, title="Part D - Single Hidden Layer: Epoch vs Loss (PCA 99%)")
plot_loss_curves(two_losses_D,    title="Part D - Two Hidden Layers: Epoch vs Loss (PCA 99%)")

### D.5 Best Model Selection and Test Evaluation

In [ ]:
print("--- Single Hidden Layer Validation Accuracies (Part D) ---")
for name, acc in single_val_accs_D.items():
    print(f"  {name}: {acc*100:.2f}%")
best_single_D = max(single_val_accs_D, key=single_val_accs_D.get)
print(f"\n  Best: {best_single_D} -> {single_val_accs_D[best_single_D]*100:.2f}%")

print("\n--- Two Hidden Layer Validation Accuracies (Part D) ---")
for name, acc in two_val_accs_D.items():
    print(f"  {name}: {acc*100:.2f}%")
best_two_D = max(two_val_accs_D, key=two_val_accs_D.get)
print(f"\n  Best: {best_two_D} -> {two_val_accs_D[best_two_D]*100:.2f}%")

In [ ]:
_, preds_single_D = evaluate_model(single_models_D[best_single_D], X_test_D, y_test_D)
print_metrics(y_test_D.numpy(), preds_single_D, dataset_name=f"Part D - {best_single_D} (Single Layer)")

_, preds_two_D = evaluate_model(two_models_D[best_two_D], X_test_D, y_test_D)
print_metrics(y_test_D.numpy(), preds_two_D, dataset_name=f"Part D - {best_two_D} (Two Layers)")

---
# Part E: Comparison Table - Test Accuracy Across All Datasets and Architectures

Now we summarize test accuracies for ALL architectures (not just the best) across all 4 datasets.

In [ ]:
# ---- collect test accuracies for each architecture ----

# Part A - single
acc_A_s1, _ = evaluate_model(single_models_A['Arch1 (561)'],  X_test_A, y_test_A)
acc_A_s2, _ = evaluate_model(single_models_A['Arch2 (1024)'], X_test_A, y_test_A)
acc_A_s3, _ = evaluate_model(single_models_A['Arch3 (128)'],  X_test_A, y_test_A)

# Part A - two
acc_A_t1, _ = evaluate_model(two_models_A['Arch1 (561,561)'],  X_test_A, y_test_A)
acc_A_t2, _ = evaluate_model(two_models_A['Arch2 (1024,128)'], X_test_A, y_test_A)
acc_A_t3, _ = evaluate_model(two_models_A['Arch3 (1024,64)'],  X_test_A, y_test_A)
acc_A_t4, _ = evaluate_model(two_models_A['Arch4 (256,64)'],   X_test_A, y_test_A)

# Part B - single
acc_B_s1, _ = evaluate_model(single_models_B['Arch1 (10)'],  X_test_B, y_test_B)
acc_B_s2, _ = evaluate_model(single_models_B['Arch2 (100)'], X_test_B, y_test_B)
acc_B_s3, _ = evaluate_model(single_models_B['Arch3 (64)'],  X_test_B, y_test_B)
acc_B_s4, _ = evaluate_model(single_models_B['Arch4 (512)'], X_test_B, y_test_B)

# Part B - two
acc_B_t1, _ = evaluate_model(two_models_B['Arch1 (64,16)'],   X_test_B, y_test_B)
acc_B_t2, _ = evaluate_model(two_models_B['Arch2 (128,32)'],  X_test_B, y_test_B)
acc_B_t3, _ = evaluate_model(two_models_B['Arch3 (256,64)'],  X_test_B, y_test_B)
acc_B_t4, _ = evaluate_model(two_models_B['Arch4 (512,128)'], X_test_B, y_test_B)

# Part C - single
acc_C_s1, _ = evaluate_model(single_models_C['Arch1 (10)'],  X_test_C, y_test_C)
acc_C_s2, _ = evaluate_model(single_models_C['Arch2 (100)'], X_test_C, y_test_C)
acc_C_s3, _ = evaluate_model(single_models_C['Arch3 (64)'],  X_test_C, y_test_C)
acc_C_s4, _ = evaluate_model(single_models_C['Arch4 (512)'], X_test_C, y_test_C)

# Part C - two
acc_C_t1, _ = evaluate_model(two_models_C['Arch1 (64,16)'],   X_test_C, y_test_C)
acc_C_t2, _ = evaluate_model(two_models_C['Arch2 (128,32)'],  X_test_C, y_test_C)
acc_C_t3, _ = evaluate_model(two_models_C['Arch3 (256,64)'],  X_test_C, y_test_C)
acc_C_t4, _ = evaluate_model(two_models_C['Arch4 (512,128)'], X_test_C, y_test_C)

# Part D - single
acc_D_s1, _ = evaluate_model(single_models_D['Arch1 (156)'], X_test_D, y_test_D)
acc_D_s2, _ = evaluate_model(single_models_D['Arch2 (512)'], X_test_D, y_test_D)
acc_D_s3, _ = evaluate_model(single_models_D['Arch3 (64)'],  X_test_D, y_test_D)

# Part D - two
acc_D_t1, _ = evaluate_model(two_models_D['Arch1 (156,156)'], X_test_D, y_test_D)
acc_D_t2, _ = evaluate_model(two_models_D['Arch2 (512,64)'],  X_test_D, y_test_D)
acc_D_t3, _ = evaluate_model(two_models_D['Arch3 (256,64)'],  X_test_D, y_test_D)
acc_D_t4, _ = evaluate_model(two_models_D['Arch4 (64,32)'],   X_test_D, y_test_D)

print("Test accuracies collected!")

In [ ]:
# ---- Build summary table ----
# For Parts B, C, D there are 4 single-layer architectures,
# but Part A only has 3 and Part D only has 3 single-layer.
# We use NaN where an architecture doesn't exist.

def pct(x):
    return round(x * 100, 2)

data = {
    "Dataset": ["Original", "Scaled", "PCA All", "PCA 99"],

    # single hidden layer models
    "Single Arch1 (%)": [pct(acc_A_s1), pct(acc_B_s1), pct(acc_C_s1), pct(acc_D_s1)],
    "Single Arch2 (%)": [pct(acc_A_s2), pct(acc_B_s2), pct(acc_C_s2), pct(acc_D_s2)],
    "Single Arch3 (%)": [pct(acc_A_s3), pct(acc_B_s3), pct(acc_C_s3), pct(acc_D_s3)],
    "Single Arch4 (%)": [None,          pct(acc_B_s4), pct(acc_C_s4), None         ],

    # two hidden layer models
    "Two Arch1 (%)": [pct(acc_A_t1), pct(acc_B_t1), pct(acc_C_t1), pct(acc_D_t1)],
    "Two Arch2 (%)": [pct(acc_A_t2), pct(acc_B_t2), pct(acc_C_t2), pct(acc_D_t2)],
    "Two Arch3 (%)": [pct(acc_A_t3), pct(acc_B_t3), pct(acc_C_t3), pct(acc_D_t3)],
    "Two Arch4 (%)": [pct(acc_A_t4), pct(acc_B_t4), pct(acc_C_t4), pct(acc_D_t4)],
}

df_summary = pd.DataFrame(data)
df_summary.set_index("Dataset", inplace=True)

print("\nTest Accuracy Comparison (%)")
print(df_summary.to_string())

In [ ]:
# ---- Visualize the summary table as a heatmap ----
plt.figure(figsize=(14, 4))
sns.heatmap(
    df_summary.astype(float),
    annot=True, fmt='.2f', cmap='YlGnBu',
    linewidths=0.5, linecolor='gray',
    cbar_kws={'label': 'Accuracy (%)'}
)
plt.title('Test Accuracy Comparison Across Datasets and Architectures (%)', fontsize=13)
plt.tight_layout()
plt.show()

## Summary / Observations

Write your observations below after running the code and seeing the results.

**1. Effect of dataset preprocessing:**
- Standardized (scaled) data generally helps neural networks converge faster and to better solutions compared to raw data, since features are all on a similar scale.
- PCA with all components should give similar results to scaled data since it's just a rotation.
- PCA with 99% variance (156 components) reduces the input dimension from 561 to 156 which should speed up training.

**2. Effect of architecture depth and width:**
- Wider hidden layers tend to have more expressive power but can overfit or take longer to converge.
- Adding a second hidden layer usually helps if the dataset has complex decision boundaries.
- For this HAR dataset, simpler architectures sometimes work well because the features are already well-engineered.

**3. Convergence:**
- Early stopping (patience=10, threshold=1e-3) prevents unnecessary epochs.
- Some architectures converge quickly (especially scaled data), while others might need more epochs.

*(Fill in specific numbers after running the experiments)*